# Intermediate 02 — State, Checkpoint, and Durable-Execution Security

A Northwind case-resolution workflow pauses before one approved provider effect. You will preserve useful resume behavior while proving that a checkpoint is evidence—not authority. The lab is synthetic, credential-free, and performs no network calls.

## Learning objectives and safety contract

By the end, you will be able to verify a complete checkpoint chain, reauthorize current state, migrate deterministically, claim one resume lease, protect the effect boundary, reconcile unknown provider outcomes, and evaluate safety separately from availability. Never place secrets or raw credentials in serialized workflow context.

In [ ]:
from dataclasses import replace
from datetime import timedelta
from pathlib import Path
import sys

COURSE = Path.cwd()
if not (COURSE / 'lab.py').exists():
    COURSE = Path('curriculum/roadmap/intermediate/02-state-checkpoint-and-durable-execution-security').resolve()
assert (COURSE / 'lab.py').exists(), 'Run from the repository root or the course directory.'
sys.path.insert(0, str(COURSE))
import lab

print('Loaded course:', COURSE.name)
print('Synthetic clock:', lab.NOW.isoformat())

## 1. Classify the persisted state

| State | Security question |
| --- | --- |
| Conversation history | What may be retained and reconstructed into context? |
| Semantic memory | Is the fact scoped, consented, current, and deletable? |
| Workflow checkpoint | Can exactly one current, authorized transition resume? |
| Event history | Can replay reconstruct deterministic decisions? |
| Provider receipt | Did the external effect actually commit? |

A checkpoint can reference other state, but it cannot turn that state into authority.

In [ ]:
service, checkpoint, effect = lab.build_system(workflow_version='workflow-v2', schema_version=2)
contract = {
    'identity': (checkpoint.run_id, checkpoint.tenant, checkpoint.owner_principal),
    'versions': (checkpoint.workflow_version, checkpoint.schema_version, checkpoint.policy_version, checkpoint.tool_version),
    'effect': (effect.operation_id, effect.effect_digest[:12], checkpoint.approval_id),
    'lifecycle': (checkpoint.status.value, checkpoint.sequence, checkpoint.remaining_steps, checkpoint.expires_at.isoformat()),
    'integrity': (checkpoint.key_id, checkpoint.payload_digest[:12], checkpoint.signature[:12]),
}
contract

## 2. Establish a deliberately unsafe baseline

The baseline below treats `paused` as sufficient permission. It ignores tenant, integrity, expiry, versions, approval binding, and concurrency. We keep it only to prove the attack suite is sensitive.

In [ ]:
def unsafe_resume(snapshot):
    return snapshot.status is lab.RunStatus.PAUSED

tampered = replace(checkpoint, tenant='south', state={'case_id': 'case-999'})
assert unsafe_resume(tampered)
print('Unsafe baseline accepted edited tenant and state:', unsafe_resume(tampered))

In [ ]:
service.store.records[checkpoint.checkpoint_id] = tampered
decision = service.resume(
    lab.actor('notebook:tamper'), checkpoint.checkpoint_id,
    resume_request_id='resume:tamper', now=lab.NOW,
)
assert (decision.status, decision.reason) == (lab.ResumeStatus.DENY, 'checkpoint-integrity')
decision.status.value, decision.reason

## 3. Migrate durable state explicitly

A deployment cannot assume old serialized state matches new workflow code. The course registers one exact version transition and runs the migration twice to detect nondeterminism.

In [ ]:
service, old_checkpoint, _ = lab.build_system()  # workflow-v1/schema-1; current is workflow-v2
migrated = service.resume(
    lab.actor('notebook:migrate'), old_checkpoint.checkpoint_id,
    resume_request_id='resume:migrate', now=lab.NOW,
)
assert migrated.status is lab.ResumeStatus.READY
assert migrated.lease.target_schema_version == 2
migrated.reason, migrated.lease.state

In [ ]:
service, checkpoint, _ = lab.build_system(workflow_version='workflow-v2', schema_version=2)
first = service.resume(lab.actor('notebook:race-1'), checkpoint.checkpoint_id, resume_request_id='resume:first', now=lab.NOW)
same = service.resume(lab.actor('notebook:race-1'), checkpoint.checkpoint_id, resume_request_id='resume:first', now=lab.NOW)
rival = service.resume(lab.actor('notebook:race-2'), checkpoint.checkpoint_id, resume_request_id='resume:rival', now=lab.NOW)
assert (first.status, same.status, rival.status) == (lab.ResumeStatus.READY, lab.ResumeStatus.IDEMPOTENT, lab.ResumeStatus.CONFLICT)
[(item.status.value, item.reason) for item in (first, same, rival)]

## 4. Recheck immediately before the effect

Resume authorization and effect authorization are separated in time. A tool contract or policy can change after a lease is granted, so the effect boundary validates current state again.

In [ ]:
service, checkpoint, _ = lab.build_system(workflow_version='workflow-v2', schema_version=2)
resumed = service.resume(lab.actor('notebook:effect'), checkpoint.checkpoint_id, resume_request_id='resume:effect', now=lab.NOW)
service.policy.current = replace(service.policy.current, tool_version='case-tool-v4')
effect_decision = service.execute_pending(lab.actor('notebook:effect'), resumed.lease, now=lab.NOW)
assert (effect_decision.status, service.provider.dispatch_count) == (lab.EffectStatus.DENY, 0)
effect_decision.status.value, effect_decision.reason, service.provider.dispatch_count

## 5. Reconcile an unknown outcome; do not blindly retry

The provider commits, but the caller observes a timeout. The stable operation ID lets the next lease query the original outcome. Approval proves permission for the original attempt, not whether it succeeded.

In [ ]:
service, checkpoint, _ = lab.build_system(workflow_version='workflow-v2', schema_version=2)
resumed = service.resume(lab.actor('notebook:unknown'), checkpoint.checkpoint_id, resume_request_id='resume:unknown', now=lab.NOW)
unknown = service.execute_pending(lab.actor('notebook:unknown'), resumed.lease, now=lab.NOW, timeout_after_provider_commit=True)
assert unknown.status is lab.EffectStatus.UNKNOWN
reopened = service.resume(lab.actor('notebook:reconcile'), unknown.checkpoint.checkpoint_id, resume_request_id='resume:reconcile', now=lab.NOW + timedelta(seconds=1))
reconciled = service.reconcile_unknown(lab.actor('notebook:reconcile'), reopened.lease, now=lab.NOW + timedelta(seconds=1))
assert reconciled.status is lab.EffectStatus.RECONCILED
assert service.provider.dispatch_count == 1
reconciled.status.value, reconciled.reason, service.provider.dispatch_count

In [ ]:
service, checkpoint, _ = lab.build_system(workflow_version='workflow-v2', schema_version=2)
service.policy.available = False
failure = service.resume(lab.actor('notebook:outage'), checkpoint.checkpoint_id, resume_request_id='resume:outage', now=lab.NOW)
assert (failure.status, failure.reason) == (lab.ResumeStatus.ERROR, 'policy-unavailable')
print('Outage preserved as:', failure.status.value, '—', failure.reason)

## 6. Evaluate safety, utility, and failures separately

A deny-all system can have a zero unsafe-resume rate and no utility. Dependency failures are missing evidence, not successful defenses. The report therefore declares populations and denominators.

In [ ]:
report, observations = lab.evaluate_controls()
assert report.valid_resume_success_rate == 1.0
assert report.unsafe_resume_rate == 0.0
assert report.duplicate_effect_count == 0
assert report.trace_completeness_rate == 1.0
report

## 7. Put a real SDK state behind the application gate

The adapter uses the repository-pinned OpenAI Agents SDK. It serializes a real `RunState`, keeps the complete blob server-side, verifies the application checkpoint first, then calls `RunState.from_json()`. This is a mechanics proof, not a model call.

In [ ]:
import sdk_adapter
sdk_proof = await sdk_adapter.credential_free_demo_async()
assert sdk_proof['resume_status'] == 'ready'
assert sdk_proof['snapshot_kept_server_side']
assert sdk_proof['tampered_blob_rejected']
assert sdk_proof['serialized_context_contains_no_credentials']
sdk_proof

## Production challenge

Replace the in-memory lease with a transactional compare-and-swap plus fencing token. Simulate worker A losing its lease, worker B acquiring a new lease, and worker A attempting a late commit. Your test must prove that the current head remains worker B's and that worker A's provider effect cannot be dispatched. Then add authenticated callback envelopes and crash injection at each persistence boundary.

## Checkpoint

A signed checkpoint has an unexpired approval, but the original provider call timed out after possible commit. Should the runtime dispatch again? **No.** Reauthorize the resume, acquire one exclusive lease, and reconcile the same operation ID. A checkpoint and approval do not prove the provider outcome.